In [2]:
import os, random, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score
from imblearn.metrics import specificity_score
from thop import profile

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [3]:
class TransformerEncoder(nn.Module):
    def __init__(self, d_model=32, n_layers=2, n_heads=4, dim_feedforward=None):
        super().__init__()
        if dim_feedforward is None:
            dim_feedforward = d_model * 4
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_feedforward,
            batch_first=True, activation='gelu'
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.final_norm = nn.LayerNorm(d_model)

    def forward(self, tokens, key_padding_mask=None):
        return self.final_norm(self.encoder(tokens, src_key_padding_mask=key_padding_mask))

In [4]:
class ROIPatchEmbed3D(nn.Module):
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32):
        super().__init__()
        self.n_rois = n_rois
        self.patch_size = patch_size
        self.grid_size = roi_size // patch_size
        self.patches_per_roi = self.grid_size ** 3
        self.d_model = d_model
        self.patch_conv = nn.Conv3d(1, d_model, kernel_size=patch_size, stride=patch_size)
        self.roi_embed = nn.Embedding(n_rois, d_model)
        self.depth_embed = nn.Embedding(self.grid_size, d_model)
        self.height_embed = nn.Embedding(self.grid_size, d_model)
        self.width_embed = nn.Embedding(self.grid_size, d_model)
        with torch.no_grad():
            for emb in [self.roi_embed, self.depth_embed, self.height_embed, self.width_embed]:
                emb.weight.mul_(0.02)
        d, h, w = torch.meshgrid(torch.arange(self.grid_size), torch.arange(self.grid_size),
                                  torch.arange(self.grid_size), indexing="ij")
        self.register_buffer("coordinates", torch.stack([d, h, w], dim=-1).reshape(-1, 3), persistent=False)

    def forward(self, rois):
        batch_size, n_rois = rois.shape[:2]
        x = rois.reshape(batch_size * n_rois, 1, rois.shape[-3], rois.shape[-2], rois.shape[-1])
        tokens = self.patch_conv(x).flatten(2).transpose(1, 2)
        tokens = tokens.reshape(batch_size, n_rois, self.patches_per_roi, self.d_model)
        coords = self.coordinates
        spatial = self.depth_embed(coords[:, 0]) + self.height_embed(coords[:, 1]) + self.width_embed(coords[:, 2])
        tokens = tokens + spatial[None, None, :, :] + self.roi_embed.weight[None, :, None, :]
        occupancy = F.max_pool3d((x.abs() > 1e-6).float(), kernel_size=self.patch_size, stride=self.patch_size)
        valid = occupancy.flatten(1).bool().reshape(batch_size, n_rois, self.patches_per_roi)
        tokens = tokens.reshape(batch_size, -1, self.d_model)
        valid = valid.reshape(batch_size, -1)
        tokens = tokens * valid.unsqueeze(-1).to(tokens.dtype)
        return tokens, valid


class TransformerBranch(nn.Module):
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32, n_layers=2, n_heads=4):
        super().__init__()
        self.n_rois = n_rois
        self.patch_embed = ROIPatchEmbed3D(n_rois, roi_size, patch_size, d_model)
        self.encoder = TransformerEncoder(d_model, n_layers, n_heads)

    def forward(self, rois):
        tokens, valid = self.patch_embed(rois)
        key_padding_mask = ~valid
        tokens = self.encoder(tokens, key_padding_mask=key_padding_mask)
        w = valid.unsqueeze(-1).to(tokens.dtype)
        pooled = (tokens * w).sum(dim=1) / w.sum(dim=1).clamp_min(1.0)
        return pooled


class TransformerVisionModel(nn.Module):
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32, n_layers=2, n_heads=4, n_classes=2, dropout=0.4):
        super().__init__()
        self.branch = TransformerBranch(n_rois, roi_size, patch_size, d_model, n_layers, n_heads)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, n_classes)

    def forward(self, rois):
        pooled = self.branch(rois)
        return self.classifier(self.dropout(pooled))


class MultimodalTransformerVisionModel(nn.Module):
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32, n_layers=2, n_heads=4, n_classes=2, dropout=0.4):
        super().__init__()
        self.mri_branch = TransformerBranch(n_rois, roi_size, patch_size, d_model, n_layers, n_heads)
        self.pet_branch = TransformerBranch(n_rois, roi_size, patch_size, d_model, n_layers, n_heads)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model * 2, n_classes)

    def forward(self, mri_rois, pet_rois):
        mri_pooled = self.mri_branch(mri_rois)
        pet_pooled = self.pet_branch(pet_rois)
        fused = torch.cat([mri_pooled, pet_pooled], dim=1)
        return self.classifier(self.dropout(fused))

In [5]:
COHORT_CSV     = "D:/mamba_model/thesis_cohort_final.csv"
MRI_CACHE_AUG  = "D:/mamba_model/preprocessed_cache_roi64_aug"
PET_CACHE_AUG  = "D:/mamba_model/preprocessed_cache_pet_aug"
CKPT_DIR       = "D:/mamba_model/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

df = pd.read_csv(COHORT_CSV)
sessions, labels = df["mri_session"].values, df["outcome_label"].values
X_tv, X_test, y_tv, y_test = train_test_split(sessions, labels, test_size=0.2, random_state=42, stratify=labels)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.25, random_state=42, stratify=y_tv)
session_to_subject = dict(zip(df["mri_session"], df["subject_id"]))

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Train: 126 | Val: 42 | Test: 42


In [6]:
class ROIDataset(Dataset):
    def __init__(self, sessions, labels, cache_dir, is_mri=True, is_train=False):
        self.samples, self.cache_dir = [], cache_dir
        for session_id, label in zip(sessions, labels):
            key = session_id if is_mri else session_to_subject[session_id]
            self.samples.append((key, label, "orig"))
            if is_train:
                for seed in [1, 101, 42]:
                    self.samples.append((key, label, f"aug{seed}"))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        key, label, version = self.samples[idx]
        rois = np.array(np.load(f"{self.cache_dir}/{key}_{version}.npy", mmap_mode="r"), dtype=np.float32, copy=True)
        return torch.from_numpy(rois).unsqueeze(1), torch.tensor(label, dtype=torch.long), key


class MultimodalROIDataset(Dataset):
    def __init__(self, sessions, labels, mri_cache_dir, pet_cache_dir, is_train=False):
        self.samples = []
        self.mri_cache_dir, self.pet_cache_dir = mri_cache_dir, pet_cache_dir
        for session_id, label in zip(sessions, labels):
            subject_id = session_to_subject[session_id]
            self.samples.append((session_id, subject_id, label, "orig"))
            if is_train:
                for seed in [1, 101, 42]:
                    self.samples.append((session_id, subject_id, label, f"aug{seed}"))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        mri_key, pet_key, label, version = self.samples[idx]
        mri_rois = np.array(np.load(f"{self.mri_cache_dir}/{mri_key}_{version}.npy", mmap_mode="r"), dtype=np.float32, copy=True)
        pet_rois = np.array(np.load(f"{self.pet_cache_dir}/{pet_key}_{version}.npy", mmap_mode="r"), dtype=np.float32, copy=True)
        return torch.from_numpy(mri_rois).unsqueeze(1), torch.from_numpy(pet_rois).unsqueeze(1), torch.tensor(label, dtype=torch.long), mri_key

In [7]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for rois, labels, _ in loader:
        rois, labels = rois.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(rois), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, preds_all, labels_all = 0, [], []
    with torch.no_grad():
        for rois, labels, _ in loader:
            rois, labels = rois.to(device), labels.to(device)
            out = model(rois)
            total_loss += criterion(out, labels).item()
            preds_all.extend(out.argmax(1).cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
    acc = np.mean(np.array(preds_all) == np.array(labels_all))
    tpr = recall_score(labels_all, preds_all, zero_division=0)
    tnr = specificity_score(labels_all, preds_all)
    return total_loss / len(loader), acc, tpr, tnr

def train_epoch_mm(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for mri_rois, pet_rois, labels, _ in loader:
        mri_rois, pet_rois, labels = mri_rois.to(device), pet_rois.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(mri_rois, pet_rois), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate_mm(model, loader, criterion, device):
    model.eval()
    total_loss, preds_all, labels_all = 0, [], []
    with torch.no_grad():
        for mri_rois, pet_rois, labels, _ in loader:
            mri_rois, pet_rois, labels = mri_rois.to(device), pet_rois.to(device), labels.to(device)
            out = model(mri_rois, pet_rois)
            total_loss += criterion(out, labels).item()
            preds_all.extend(out.argmax(1).cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
    acc = np.mean(np.array(preds_all) == np.array(labels_all))
    tpr = recall_score(labels_all, preds_all, zero_division=0)
    tnr = specificity_score(labels_all, preds_all)
    return total_loss / len(loader), acc, tpr, tnr

In [8]:
def measure_inference_time(model, loader, device, is_multimodal, n_batches=20):
    model.eval()
    times = []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= n_batches: break
            if is_multimodal:
                mri_rois, pet_rois, labels, _ = batch
                mri_rois, pet_rois = mri_rois.to(device), pet_rois.to(device)
                bs = mri_rois.shape[0]
                if device.type == 'cuda': torch.cuda.synchronize()
                t0 = time.time()
                _ = model(mri_rois, pet_rois)
            else:
                rois, labels, _ = batch
                rois = rois.to(device)
                bs = rois.shape[0]
                if device.type == 'cuda': torch.cuda.synchronize()
                t0 = time.time()
                _ = model(rois)
            if device.type == 'cuda': torch.cuda.synchronize()
            times.append((time.time() - t0) / bs)
    return np.mean(times), np.std(times)

def try_compute_flops(model, loader, device, is_multimodal):
    try:
        model.eval()
        batch = next(iter(loader))
        with torch.no_grad():
            if is_multimodal:
                mri_rois, pet_rois, labels, _ = batch
                inputs = (mri_rois[:1].to(device), pet_rois[:1].to(device))
            else:
                rois, labels, _ = batch
                inputs = (rois[:1].to(device),)
            macs, _ = profile(model, inputs=inputs, verbose=False)
        return macs * 2
    except Exception as e:
        print(f"  (FLOPs failed: {e})")
        return None

def run_one_seed(seed, model_class, train_loader, val_loader, test_loader, is_multimodal, save_prefix, n_rois=6, max_epochs=101, patience=15):
    torch.manual_seed(seed); torch.cuda.manual_seed(seed); np.random.seed(seed); random.seed(seed)

    model = model_class(n_rois=n_rois, d_model=32, n_layers=2, n_heads=4, n_classes=2, dropout=0.4).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
    train_fn = train_epoch_mm if is_multimodal else train_epoch
    eval_fn = evaluate_mm if is_multimodal else evaluate

    best_val_loss, no_improve, best_epoch, total_time = float("inf"), 0, 0, 0
    save_path = f"{CKPT_DIR}/{save_prefix}_seed{seed}.pt"

    print(f"\n--- Seed {seed} ---")
    print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>10} | {'Val Acc':>8} | {'Val TPR':>8} | {'Val TNR':>8} | {'Time':>6}")
    print("-" * 70)

    for epoch in range(1, max_epochs):
        t0 = time.time()
        train_loss = train_fn(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc, val_tpr, val_tnr = eval_fn(model, val_loader, criterion, device)
        scheduler.step(val_loss)
        epoch_time = time.time() - t0
        total_time += epoch_time
        print(f"{epoch:>6} | {train_loss:>10.4f} | {val_loss:>10.4f} | {val_acc:>8.4f} | {val_tpr:>8.4f} | {val_tnr:>8.4f} | {epoch_time:>5.1f}s")

        if val_loss < best_val_loss:
            best_val_loss, best_epoch, no_improve = val_loss, epoch, 0
            torch.save(model.state_dict(), save_path)
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch}. Best: {best_epoch}")
                break

    model.load_state_dict(torch.load(save_path, weights_only=True))
    test_loss, test_acc, test_tpr, test_tnr = eval_fn(model, test_loader, criterion, device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    inf_mean, inf_std = measure_inference_time(model, test_loader, device, is_multimodal)
    flops = try_compute_flops(model, test_loader, device, is_multimodal)

    print(f"\n  >>> Seed {seed} TEST: Acc={test_acc*100:.1f}% | TPR={test_tpr*100:.1f}% | TNR={test_tnr*100:.1f}% | "
          f"train_time={total_time/60:.1f}min | inf={inf_mean*1000:.2f}ms | {f'{flops/1e9:.2f}GFLOPs' if flops else 'N/A'}")

    return {"seed": seed, "acc": test_acc, "tpr": test_tpr, "tnr": test_tnr, "best_epoch": best_epoch,
            "train_time_sec": total_time, "n_params": n_params, "inf_time_ms": inf_mean * 1000, "flops": flops}

In [8]:
BATCH_SIZE = 4
mri_train_loader = DataLoader(ROIDataset(X_train, y_train, MRI_CACHE_AUG, True, True), batch_size=BATCH_SIZE, shuffle=True)
mri_val_loader   = DataLoader(ROIDataset(X_val, y_val, MRI_CACHE_AUG, True, False), batch_size=BATCH_SIZE, shuffle=False)
mri_test_loader  = DataLoader(ROIDataset(X_test, y_test, MRI_CACHE_AUG, True, False), batch_size=BATCH_SIZE, shuffle=False)

print("=== TRANSFORMER, MRI-ONLY: 3-seed run ===")
transformer_mri_results = [run_one_seed(s, TransformerVisionModel, mri_train_loader, mri_val_loader, mri_test_loader,
                                          False, "transformer_mri", n_rois=6) for s in [1, 7, 123]]

=== TRANSFORMER, MRI-ONLY: 3-seed run ===

--- Seed 1 ---
 Epoch | Train Loss |   Val Loss |  Val Acc |  Val TPR |  Val TNR |   Time
----------------------------------------------------------------------
     1 |     0.7004 |     0.6912 |   0.5476 |   0.8571 |   0.2381 |  49.8s
     2 |     0.7002 |     0.6923 |   0.5000 |   1.0000 |   0.0000 |  10.0s
     3 |     0.6916 |     0.6878 |   0.5238 |   1.0000 |   0.0476 |   9.9s
     4 |     0.6960 |     0.6864 |   0.5476 |   0.1429 |   0.9524 |  10.0s
     5 |     0.6901 |     0.6844 |   0.6190 |   0.5714 |   0.6667 |  10.1s
     6 |     0.6872 |     0.6917 |   0.5238 |   0.0476 |   1.0000 |  10.1s
     7 |     0.6869 |     0.6957 |   0.5000 |   0.0000 |   1.0000 |  10.2s
     8 |     0.6871 |     0.6864 |   0.5476 |   0.0952 |   1.0000 |  10.2s
     9 |     0.6793 |     0.6755 |   0.6667 |   0.7619 |   0.5714 |   9.8s
    10 |     0.6777 |     0.6711 |   0.6667 |   0.7619 |   0.5714 |   9.7s
    11 |     0.6603 |     0.6684 |   0.5476 | 

In [10]:
BATCH_SIZE = 4
pet_train_loader = DataLoader(ROIDataset(X_train, y_train, PET_CACHE_AUG, False, True), batch_size=BATCH_SIZE, shuffle=True)
pet_val_loader   = DataLoader(ROIDataset(X_val, y_val, PET_CACHE_AUG, False, False), batch_size=BATCH_SIZE, shuffle=False)
pet_test_loader  = DataLoader(ROIDataset(X_test, y_test, PET_CACHE_AUG, False, False), batch_size=BATCH_SIZE, shuffle=False)

print("=== TRANSFORMER, PET-ONLY: 3-seed run ===")
transformer_pet_results = [run_one_seed(s, TransformerVisionModel, pet_train_loader, pet_val_loader, pet_test_loader,
                                          False, "transformer_pet", n_rois=6) for s in [1, 7, 123]]

=== TRANSFORMER, PET-ONLY: 3-seed run ===

--- Seed 1 ---
 Epoch | Train Loss |   Val Loss |  Val Acc |  Val TPR |  Val TNR |   Time
----------------------------------------------------------------------
     1 |     0.7027 |     0.6906 |   0.5000 |   1.0000 |   0.0000 |  56.1s
     2 |     0.7007 |     0.6958 |   0.5000 |   1.0000 |   0.0000 |  10.4s
     3 |     0.6902 |     0.6895 |   0.5000 |   1.0000 |   0.0000 |   9.8s
     4 |     0.6932 |     0.6882 |   0.5714 |   0.3333 |   0.8095 |   9.5s
     5 |     0.6890 |     0.6866 |   0.5714 |   0.7143 |   0.4286 |   9.6s
     6 |     0.6887 |     0.6882 |   0.5476 |   0.0952 |   1.0000 |   9.5s
     7 |     0.6852 |     0.6892 |   0.5238 |   0.0952 |   0.9524 |  10.1s
     8 |     0.6863 |     0.6855 |   0.5238 |   0.0476 |   1.0000 |  10.2s
     9 |     0.6822 |     0.6854 |   0.5000 |   0.8571 |   0.1429 |   9.7s
    10 |     0.6837 |     0.6835 |   0.5238 |   0.8571 |   0.1905 |  10.3s
    11 |     0.6747 |     0.6859 |   0.5238 | 

In [11]:
mm_train_loader = DataLoader(MultimodalROIDataset(X_train, y_train, MRI_CACHE_AUG, PET_CACHE_AUG, True), batch_size=4, shuffle=True)
mm_val_loader   = DataLoader(MultimodalROIDataset(X_val, y_val, MRI_CACHE_AUG, PET_CACHE_AUG, False), batch_size=4, shuffle=False)
mm_test_loader  = DataLoader(MultimodalROIDataset(X_test, y_test, MRI_CACHE_AUG, PET_CACHE_AUG, False), batch_size=4, shuffle=False)

print("=== TRANSFORMER, MULTIMODAL: 3-seed run ===")
transformer_mm_results = [run_one_seed(s, MultimodalTransformerVisionModel, mm_train_loader, mm_val_loader, mm_test_loader,
                                        True, "transformer_mm", n_rois=6) for s in [1, 7, 123]]

=== TRANSFORMER, MULTIMODAL: 3-seed run ===

--- Seed 1 ---
 Epoch | Train Loss |   Val Loss |  Val Acc |  Val TPR |  Val TNR |   Time
----------------------------------------------------------------------
     1 |     0.7016 |     0.6890 |   0.6667 |   0.5238 |   0.8095 |  65.5s
     2 |     0.6945 |     0.6887 |   0.5238 |   0.0952 |   0.9524 | 239.5s
     3 |     0.6924 |     0.6836 |   0.5238 |   0.0952 |   0.9524 | 306.4s
     4 |     0.6841 |     0.6782 |   0.5000 |   1.0000 |   0.0000 | 316.2s
     5 |     0.6890 |     0.6843 |   0.5476 |   0.0952 |   1.0000 | 299.1s
     6 |     0.6904 |     0.6740 |   0.5476 |   0.0952 |   1.0000 | 298.4s
     7 |     0.6702 |     0.6702 |   0.6190 |   0.4762 |   0.7619 | 184.3s
     8 |     0.6658 |     0.6663 |   0.5952 |   0.3810 |   0.8095 | 102.3s
     9 |     0.6617 |     0.6619 |   0.6429 |   0.7619 |   0.5238 | 113.9s
    10 |     0.6521 |     0.6624 |   0.5714 |   0.3810 |   0.7619 |  99.7s
    11 |     0.6416 |     0.6486 |   0.5714 

In [12]:
def summarize(results, name):
    accs, tprs, tnrs = [r["acc"] for r in results], [r["tpr"] for r in results], [r["tnr"] for r in results]
    print(f"{name}: Acc={np.mean(accs)*100:.1f}±{np.std(accs,ddof=1)*100:.1f}% | "
          f"TPR={np.mean(tprs)*100:.1f}±{np.std(tprs,ddof=1)*100:.1f}% | "
          f"TNR={np.mean(tnrs)*100:.1f}±{np.std(tnrs,ddof=1)*100:.1f}%")

print("=== Transformer (6 regions together) ===")
summarize(transformer_mri_results, "MRI-only")
summarize(transformer_pet_results, "PET-only")
summarize(transformer_mm_results, "Multimodal")

print("\n=== Compare to v4 (Mamba) ===")
print("MRI-only:   Acc=68.3±2.7% | TPR=63.5±7.3% | TNR=73.0±2.7%")
print("PET-only:   Acc=62.7±5.0% | TPR=68.3±12.0% | TNR=57.1±4.8%")
print("Multimodal: Acc=65.1±1.4% | TPR=58.7±5.5% | TNR=71.4±4.8%")

=== Transformer (6 regions together) ===


NameError: name 'transformer_mri_results' is not defined